# Flash Daily MTD — Audit & Debug (Mondelez)

Notebook tracking + debug số liệu **Flash Daily** theo phạm vi **Month-to-Date** trên ClickHouse.

**View nguồn** — `analytics_workspace.mv_flash_and_drop_report` (UNION ALL của 2 MV gốc):

| MV gốc | Vai trò | Refresh |
|---|---|---|
| `mv_flash_report` | Đơn còn sống — 5 trạng thái `e2e_label` mutually exclusive (Chưa xuất / Đang xuất / Đã xuất / Đang vận / Đã vận) | 5 phút |
| `mv_dropped_report` | Đơn đã rớt — `e2e_label = 'Kế hoạch hủy'` | (theo `mv_flash_and_drop_report`) |
| `mv_flash_and_drop_report` | Union 2 MV trên — view chính cho widget Flash Daily | 15 phút |

## Cấu trúc notebook

| Mức | Mục đích | Kỳ vọng |
|---|---|---|
| **Setup** | Kết nối CH + helper `q()` (chạy 1 lần) | — |
| **L0 · Tham số MTD** | Sửa `MTD_FROM` / `MTD_TO` / `DATE_COL` / `UOM_COL` | — |
| **L1 · Sức khỏe dữ liệu** | Row count, freshness, distinct counts | rows > 0, lag < 30 phút |
| **L2 · Phân bố** | `e2e_label`, `status`, `type`, `whseid`, `brand`, `kênh` | Tổng `e2e_label` rows = window rows |
| **L3 · Tổng volume** | Plan/Shipped/Delivered theo CSE / KG / CBM | Plan ≥ Shipped ≥ Delivered |
| **L4 · Bất thường — NULL/empty** | NULL count critical cols | Critical (so/whseid/delivery_date_1) = 0 |
| **L5 · Bất thường — Volume integrity** | Negative, shipped > plan, delivered > shipped | = 0 |
| **L6 · Bất thường — Business rule** | Cancel-but-delivered, e2e vs timestamp gap | = 0 hoặc rất nhỏ |
| **L7 · Bất thường — Key & cross-MV parity** | Duplicate `(so, orderlinenumber)`, combined ≠ flash + dropped | dup = 0, parity diff = 0 |
| **L8 · Bất thường — Timestamp ordering** | ATA < ETD, create > delivery, ... | = 0 |
| **Summary** | Dashboard 1-cell rollup L1-L8 — chạy cuối các section anomaly để biết window có cần action không | ✗ FAIL = 0 |
| **L9 · Drill listout** | 9 sub-section, mỗi cái list ≤ 100 đơn vi phạm + hotspot theo whseid/khu_vuc (L9.5 dành riêng cho `ata_before_etd`) | — |
| **L10 · Ad-hoc SQL** | Slot truy vấn tự do | — |

> **Cách dùng:** chạy Setup + L0 một lần, sau đó chạy từng cell L1-L8 lần lượt. Mỗi cell anomaly hiển thị bảng count — nếu thấy số > 0 ở cột bất thường → drill xuống bằng L9 hoặc L10.

## Setup — chạy 1 lần

In [37]:
import os
from pathlib import Path
from datetime import date, timedelta
from dotenv import load_dotenv
import pandas as pd
import clickhouse_connect
from IPython.display import display, Markdown

def _find_tenant():
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if d.name == 'mondelez' and (d / '.env').exists():
            return d
        if (d / 'mondelez' / '.env').exists():
            return d / 'mondelez'
        if (d / 'projects' / 'mondelez' / '.env').exists():
            return d / 'projects' / 'mondelez'
    return here

_TENANT = _find_tenant()
load_dotenv(_TENANT / '.env')

DB = 'analytics_workspace'
T_MAIN  = f"`{DB}`.`mv_flash_and_drop_report`"
T_FLASH = f"`{DB}`.`mv_flash_report`"
T_DROP  = f"`{DB}`.`mv_dropped_report`"

client = clickhouse_connect.get_client(
    host=os.getenv('CLICKHOUSE_HOST', ''),
    port=int(os.getenv('CLICKHOUSE_PORT', '8443')),
    username=os.getenv('CLICKHOUSE_USER', ''),
    password=os.getenv('CLICKHOUSE_PASSWORD', ''),
    secure=os.getenv('CLICKHOUSE_SECURE', 'true').lower() not in ('false', '0', 'no'),
    connect_timeout=15,
    send_receive_timeout=120,
)

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', lambda v: f'{v:,.2f}')

# Helper — render SQL với placeholder {key} thay bằng PARAMS + T_MAIN/T_FLASH/T_DROP/extras.
def q(sql: str, **extras) -> pd.DataFrame:
    rendered = sql.format(**{**PARAMS, 'T_MAIN': T_MAIN, 'T_FLASH': T_FLASH, 'T_DROP': T_DROP, **extras})
    return client.query_df(rendered)

# PARAMS — placeholder; sẽ override ở cell L0
PARAMS = {}

print('Setup OK — chuyển sang cell L0 để chỉnh tham số MTD')


Setup OK — chuyển sang cell L0 để chỉnh tham số MTD


## L0 · Tham số MTD — sửa ở đây rồi chạy

| Tham số | Mặc định | Ghi chú |
|---|---|---|
| `MTD_FROM` | ngày 01 của tháng hiện tại (HCM time) | đầu cửa sổ MTD |
| `MTD_TO` | hôm nay | cuối cửa sổ MTD (inclusive) |
| `DATE_COL` | `delivery_date_1` | cột date để filter — khớp Flash Daily Date Type "Ngày GI" |
| `DATE_LABEL` | `Ngày GI` | nhãn label (UI), để hiển thị, không tham gia filter |
| `UOM_COL` | `original_cse` | đơn vị volume hiển thị — chọn `original_cse` / `original_kg` / `original_cbm` / `original_pl` |

**Mapping Date Type → cột (per CH CASE branch trong widget Flash Daily):**

| Label UI | Cột MV |
|---|---|
| Ngày GI | `delivery_date_1` |
| Actual Ship Date | `actual_ship_date` |
| ETD gửi thầu (đơn) | `etd_chuyen_gui_thau` |
| ETA gửi thầu (đơn) | `eta_giao_hang_cho_npp` |
| ATA đơn | `ata_den` |

In [38]:
from datetime import date as _date
_today = _date.today()
MTD_FROM = _today.replace(day=1).isoformat()
MTD_TO = _today.isoformat()
DATE_COL = 'delivery_date_1'
DATE_LABEL = 'Ngày GI'
UOM_COL = 'original_cse'

PARAMS.update(
    from_date=MTD_FROM,
    to_date=MTD_TO,
    date_col=DATE_COL,
    date_label=DATE_LABEL,
    uom_col=UOM_COL,
)
print(f"Window MTD: {MTD_FROM} → {MTD_TO}")
print(f"Date column: {DATE_COL}  (label: {DATE_LABEL})")
print(f"Volume column: {UOM_COL}")


Window MTD: 2026-05-01 → 2026-05-28
Date column: delivery_date_1  (label: Ngày GI)
Volume column: original_cse


## L1 · Sức khỏe dữ liệu

3 góc nhìn:

- **A** — Tổng row mỗi MV gốc (không lọc window) — sanity check MV còn refresh.
- **B** — Distinct count trong window MTD — kiểm tra phủ data (số kho, customer, brand, ...).
- **C** — Freshness: max timestamp + lag vs `now()` ở UTC+7. Lag > 30 phút trên `delivery_date_1` là bình thường (delivery date có thể tương lai), nhưng lag `ngay_tao_don` > 1h là dấu hiệu MV đang stuck.

In [39]:
df_rows = q("""
SELECT 'mv_flash_and_drop_report' AS source_mv, count() AS rows FROM {T_MAIN}
UNION ALL
SELECT 'mv_flash_report'          AS source_mv, count() AS rows FROM {T_FLASH}
UNION ALL
SELECT 'mv_dropped_report'        AS source_mv, count() AS rows FROM {T_DROP}
""")
display(Markdown('**A · Tổng row mỗi MV gốc (không lọc window)**'))
display(df_rows)

df_distinct = q("""
SELECT
  '{date_label}'                                                       AS date_type,
  '{from_date}'                                                        AS from_date,
  '{to_date}'                                                          AS to_date,
  count()                                                              AS rows_mtd,
  uniq(so)                                                             AS distinct_so,
  uniq(coalesce(whseid, ''))                                           AS distinct_whseid,
  uniq(coalesce(customer_code, ''))                                    AS distinct_customer,
  uniq(coalesce(brand, ''))                                            AS distinct_brand,
  uniq(coalesce(group_of_cago, ''))                                    AS distinct_cargo_group,
  uniq(coalesce(group_name, ''))                                       AS distinct_kenh,
  uniq(coalesce(khu_vuc_doi_xe, ''))                                   AS distinct_khu_vuc,
  uniq(coalesce(e2e_label, ''))                                        AS distinct_e2e_label,
  uniq(coalesce(status, ''))                                           AS distinct_status,
  uniq(coalesce(type, ''))                                             AS distinct_order_type
FROM {T_MAIN}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
""")
display(Markdown('**B · Distinct count trong window MTD**'))
display(df_distinct.T.rename(columns={0: 'value'}))

df_fresh = q("""
SELECT
  toDateTime(max({date_col}),       'Asia/Ho_Chi_Minh') AS max_date_col,
  toDateTime(min({date_col}),       'Asia/Ho_Chi_Minh') AS min_date_col,
  toDateTime(max(ngay_tao_don),     'Asia/Ho_Chi_Minh') AS max_ngay_tao_don,
  toDateTime(max(actual_ship_date), 'Asia/Ho_Chi_Minh') AS max_actual_ship,
  toDateTime(max(ata_den),          'Asia/Ho_Chi_Minh') AS max_ata_den,
  now('Asia/Ho_Chi_Minh')                               AS server_now,
  dateDiff('minute', max(ngay_tao_don), now())          AS lag_minutes_ngay_tao_don,
  dateDiff('minute', max(actual_ship_date), now())      AS lag_minutes_actual_ship
FROM {T_MAIN}
""")
display(Markdown('**C · Freshness — max timestamp + lag vs `now()` (UTC+7)**'))
display(df_fresh.T.rename(columns={0: 'value'}))


**A · Tổng row mỗi MV gốc (không lọc window)**

,source_mv,rows
0,mv_flash_and_drop_report,6360463
1,mv_flash_report,6317597
2,mv_dropped_report,42866


**B · Distinct count trong window MTD**

,value
date_type,Ngày GI
from_date,2026-05-01
to_date,2026-05-28
rows_mtd,101425
distinct_so,15798
distinct_whseid,2
distinct_customer,1011
distinct_brand,10
distinct_cargo_group,3
distinct_kenh,3


**C · Freshness — max timestamp + lag vs `now()` (UTC+7)**

,value
max_date_col,2026-08-01 14:00:00+07:00
min_date_col,2021-08-03 07:00:00+07:00
max_ngay_tao_don,2026-05-28 20:01:07+07:00
max_actual_ship,2026-05-28 17:57:39+07:00
max_ata_den,2026-05-28 21:01:21+07:00
server_now,2026-05-28 14:50:47+07:00
lag_minutes_ngay_tao_don,-311
lag_minutes_actual_ship,-187


## L2 · Phân bố

- **A** — Phân bố theo `e2e_label` — canonical 5+1 status. Tổng `rows` phải = `rows_mtd` ở L1.
- **B** — Phân bố theo `status` (enum raw từ TMS), `type` (order type), `whseid`, `brand`, `kênh` — top 30 mỗi chiều.

> Nếu thấy `e2e_label = '(NULL)'` xuất hiện → bug ở MV `mv_flash_report` (status mapping miss), cần raise lên `/da-ch`.

In [40]:
df_e2e = q("""
SELECT coalesce(nullIf(e2e_label, ''), '(NULL)')              AS e2e_label,
       count()                                                AS rows,
       round(count() * 100.0 / sum(count()) OVER (), 2)       AS pct_rows,
       round(sum({uom_col}), 2)                               AS volume_uom
FROM {T_MAIN}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
GROUP BY e2e_label
ORDER BY rows DESC
""")
display(Markdown(f'**A · Phân bố theo `e2e_label` (volume tính theo `{UOM_COL}`)**'))
display(df_e2e)


**A · Phân bố theo `e2e_label` (volume tính theo `original_cse`)**

,e2e_label,rows,pct_rows,volume_uom
0,Đã vận chuyển,95508,94.17,"924,929.62"
1,Đã xuất kho,4731,4.66,"56,045.47"
2,Đang vận chuyển,774,0.76,"5,372.00"
3,Kế hoạch hủy,385,0.38,"3,144.17"
4,Chưa xuất kho,27,0.03,188.00


In [41]:
for dim, label in [
    ('status',         'status (enum TMS raw)'),
    ('type',           'type (order type)'),
    ('whseid',         'whseid (kho)'),
    ('brand',          'brand'),
    ('group_name',     'group_name (kênh)'),
    ('khu_vuc_doi_xe', 'khu_vuc_doi_xe (khu vực)'),
]:
    df = q("""
SELECT coalesce(nullIf({dim}, ''), '(NULL)')                  AS value,
       count()                                                AS rows,
       round(sum({uom_col}), 2)                               AS volume_uom
FROM {T_MAIN}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
GROUP BY value
ORDER BY rows DESC
LIMIT 30
""", dim=dim)
    display(Markdown(f'**B · Phân bố theo `{label}` — top 30**'))
    display(df)


**B · Phân bố theo `status (enum TMS raw)` — top 30**

,value,rows,volume_uom
0,ShipCompleted,101013,"986,347.09"
1,Cancel,385,"3,144.17"
2,New,27,188.00


**B · Phân bố theo `type (order type)` — top 30**

,value,rows,volume_uom
0,240,101425,"989,679.26"


**B · Phân bố theo `whseid (kho)` — top 30**

...

,value,rows,volume_uom
0,KD,41421,"274,511.27"
1,Oreo,18822,"285,825.00"
2,Slide,13719,"100,062.00"
3,Cosy,11625,"167,126.00"
4,Solite,6975,"106,999.00"
5,AFC,6136,"39,392.00"
6,Lu,1142,"3,591.00"
7,RITZ,957,"7,343.00"
8,Other,625,"4,814.99"
9,(NULL),3,15.00


**B · Phân bố theo `group_name (kênh)` — top 30**

,value,rows,volume_uom
0,GT,56767,"623,875.04"
1,MT,42095,"276,552.14"
2,KA,2563,"89,252.07"


**B · Phân bố theo `khu_vuc_doi_xe (khu vực)` — top 30**

,value,rows,volume_uom
0,(NULL),19899,"84,220.74"
1,North East - North West,14867,"148,487.30"
2,South East,12184,"134,879.21"
3,Ho Chi Minh,10649,"154,982.33"
4,Ha Noi,9336,"104,335.94"
5,North Central Coast,8695,"88,165.06"
6,Mekong 2,6181,"54,467.57"
7,Central,5590,"63,983.10"
8,Mekong 1,5141,"59,538.37"
9,South Central Coast,4003,"36,769.67"


## L3 · Tổng volume MTD (Plan / Shipped / Delivered × CSE / KG / CBM)

Sanity rule:

- `plan_*` ≥ `shipped_*` ≥ `delivered_*` cho mọi UOM (CSE/KG/CBM/PL).
- Nếu `shipped_cse > plan_cse` → bug source data (đã giao nhiều hơn kế hoạch), drill bằng L5.
- Nếu `delivered_cse > shipped_cse` → STM ghi sai (đã vận chuyển > đã xuất kho), drill bằng L6.

In [42]:
df_vol = q("""
SELECT
  round(sum(original_cse), 2)         AS plan_cse,
  round(sum(shipped_cse), 2)          AS shipped_cse,
  round(sum(san_luong_giao_cse), 2)   AS delivered_cse,
  round(sum(original_kg), 2)          AS plan_kg,
  round(sum(shipped_kg), 2)           AS shipped_kg,
  round(sum(san_luong_giao_kg), 2)    AS delivered_kg,
  round(sum(original_cbm), 2)         AS plan_cbm,
  round(sum(shipped_cbm), 2)          AS shipped_cbm,
  round(sum(san_luong_giao_cbm), 2)   AS delivered_cbm,
  round(sum(original_pl), 2)          AS plan_pl,
  round(sum(shipped_pl), 2)           AS shipped_pl,
  round(sum(san_luong_giao_pl), 2)    AS delivered_pl
FROM {T_MAIN}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
""")
display(Markdown('**Tổng volume MTD theo Plan/Shipped/Delivered × CSE/KG/CBM/PL**'))
display(df_vol.T.rename(columns={0: 'value'}))

df_daily = q("""
SELECT
  toDate({date_col})                          AS day,
  count()                                     AS rows,
  round(sum(original_cse), 2)                 AS plan_cse,
  round(sum(shipped_cse), 2)                  AS shipped_cse,
  round(sum(san_luong_giao_cse), 2)           AS delivered_cse,
  round(if(sum(original_cse) > 0,
        sum(san_luong_giao_cse) / sum(original_cse) * 100, 0), 2) AS pct_done
FROM {T_MAIN}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
GROUP BY day
ORDER BY day
""")
display(Markdown('**Daily volume trong MTD window — quan sát ngày sụt/tăng đột biến**'))
display(df_daily)


**Tổng volume MTD theo Plan/Shipped/Delivered × CSE/KG/CBM/PL**

,value
plan_cse,"989,679.26"
shipped_cse,"986,347.09"
delivered_cse,"921,743.69"
plan_kg,"3,684,755.27"
shipped_kg,"3,670,560.79"
delivered_kg,"3,464,874.69"
plan_cbm,"36,299.75"
shipped_cbm,"36,120.78"
delivered_cbm,"34,390.82"
plan_pl,"24,046.71"


**Daily volume trong MTD window — quan sát ngày sụt/tăng đột biến**

,day,rows,plan_cse,shipped_cse,delivered_cse,pct_done
0,2026-05-01,24,89.00,89.00,89.00,100.00
1,2026-05-02,2236,"8,802.08","8,802.08","8,381.08",95.22
2,2026-05-04,4614,"50,396.79","50,376.79","47,480.52",94.21
3,2026-05-05,4528,"47,822.39","46,960.39","43,551.19",91.07
4,2026-05-06,8598,"69,466.13","69,336.53","66,454.60",95.66
5,2026-05-07,3211,"45,483.67","45,301.40","37,704.40",82.90
6,2026-05-08,4759,"56,793.80","56,239.50","54,133.50",95.32
7,2026-05-09,1791,"26,624.30","26,576.30","23,463.30",88.13
8,2026-05-11,5239,"45,808.28","45,641.28","42,048.28",91.79
9,2026-05-12,2932,"29,502.70","29,400.70","26,761.70",90.71


## L4 · Bất thường — NULL / Empty

Critical fields kỳ vọng = 0:

- `so`, `whseid`, `delivery_date_1` — nếu NULL → row vô dụng cho mọi report
- `original_cse` = 0 — không có kế hoạch volume → check lại schema feed
- `e2e_label` NULL — bug status mapping trong `mv_flash_report`

Acceptable > 0 nhưng cần monitor:

- `customer_code` rỗng — đơn hệ thống / trả về, business sẽ quyết định có bucket riêng không
- `khu_vuc_doi_xe` rỗng — đơn chưa lên chuyến (xe chưa gán) → kéo theo L5 panel "Khu vực" có nhóm `(rỗng)`

In [43]:
df_nulls = q("""
SELECT
  countIf(so IS NULL OR so = '')                              AS so_null_or_empty,
  countIf(whseid IS NULL OR whseid = '')                      AS whseid_null_or_empty,
  countIf(customer_code IS NULL OR customer_code = '')        AS customer_code_null_or_empty,
  countIf(customer_name IS NULL OR customer_name = '')        AS customer_name_null_or_empty,
  countIf(brand IS NULL OR brand = '')                        AS brand_null_or_empty,
  countIf(group_of_cago IS NULL OR group_of_cago = '')        AS cargo_group_null_or_empty,
  countIf(group_name IS NULL OR group_name = '')              AS kenh_null_or_empty,
  countIf(khu_vuc_doi_xe IS NULL OR khu_vuc_doi_xe = '')      AS khu_vuc_null_or_empty,
  countIf(delivery_date_1 IS NULL)                            AS delivery_date_null,
  countIf(ngay_tao_don IS NULL)                               AS ngay_tao_don_null,
  countIf(original_cse IS NULL OR original_cse = 0)           AS original_cse_zero_or_null,
  countIf(e2e_label IS NULL OR e2e_label = '')                AS e2e_label_null_or_empty,
  countIf(status IS NULL OR status = '')                      AS status_null_or_empty,
  countIf(type IS NULL OR type = '')                          AS order_type_null_or_empty,
  count()                                                     AS total_rows_in_window
FROM {T_MAIN}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
""")
total = int(df_nulls['total_rows_in_window'].iloc[0])
out = df_nulls.T.rename(columns={0: 'count'})
out['pct'] = (out['count'] / total * 100).round(3) if total else 0
display(Markdown(f'**NULL / empty count trên cột critical (window có {total:,} row)**'))
display(out)


**NULL / empty count trên cột critical (window có 101,425 row)**

,count,pct
so_null_or_empty,0,0.00
whseid_null_or_empty,0,0.00
customer_code_null_or_empty,0,0.00
customer_name_null_or_empty,0,0.00
brand_null_or_empty,3,0.00
cargo_group_null_or_empty,0,0.00
kenh_null_or_empty,0,0.00
khu_vuc_null_or_empty,19899,19.62
delivery_date_null,0,0.00
ngay_tao_don_null,0,0.00


## L5 · Bất thường — Volume integrity

Tất cả các metric trong bảng dưới **kỳ vọng = 0**. Nếu > 0 → có row vi phạm quan hệ pipeline `Plan → Shipped → Delivered`.

- `neg_*` — volume âm (lỗi source data)
- `shipped_gt_plan` — đã xuất nhiều hơn kế hoạch (master data sai hoặc đơn bị tách-gộp)
- `delivered_gt_shipped` — STM báo giao > WMS báo xuất → STM ghi sai hoặc lag delete

In [44]:
df_int = q("""
SELECT
  countIf(original_cse < 0)                                          AS neg_original_cse,
  countIf(shipped_cse < 0)                                           AS neg_shipped_cse,
  countIf(san_luong_giao_cse < 0)                                    AS neg_delivered_cse,
  countIf(original_qty < 0)                                          AS neg_original_qty,
  countIf(shipped_cse > original_cse AND original_cse > 0)           AS shipped_gt_plan,
  countIf(san_luong_giao_cse > shipped_cse AND shipped_cse > 0)      AS delivered_gt_shipped,
  countIf(original_qty <= 0 AND original_cse > 0)                    AS qty0_but_cse_positive,
  countIf(original_cse > 0 AND original_pl IS NULL)                  AS cse_positive_but_pl_null,
  countIf(san_luong_giao_cse > 0 AND ata_den IS NULL)                AS delivered_volume_but_no_ata
FROM {T_MAIN}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
""")
display(Markdown('**Volume integrity violations (kỳ vọng = 0 mọi dòng)**'))
display(df_int.T.rename(columns={0: 'violations'}))


**Volume integrity violations (kỳ vọng = 0 mọi dòng)**

,violations
neg_original_cse,0
neg_shipped_cse,0
neg_delivered_cse,0
neg_original_qty,0
shipped_gt_plan,0
delivered_gt_shipped,1323
qty0_but_cse_positive,0
cse_positive_but_pl_null,0
delivered_volume_but_no_ata,4575


## L6 · Bất thường — Business rule cross-field

Quan hệ logic giữa `status` / `e2e_label` / timestamp / volume. Mỗi metric = 0 là lý tưởng; số nhỏ (< 0.1% window) có thể chấp nhận, nhưng nếu lớn → drill bằng L9.

| Metric | Vi phạm |
|---|---|
| `cancel_but_delivered` | `status='Cancel'` nhưng `san_luong_giao_cse > 0` → đã giao mà ghi Cancel |
| `delivered_label_but_no_ata` | `e2e_label='Đã vận chuyển'` nhưng `ata_den IS NULL` → STM lag |
| `shipping_label_but_no_atd` | `e2e_label='Đang vận chuyển'` nhưng `atd_den IS NULL` → STM lag |
| `shipped_label_but_has_full_stm` | `e2e_label='Đã xuất kho'` nhưng đã có ATD + `thoi_gian_di` → MV chưa update bucket |
| `ata_set_but_still_shipping` | Đã có ATA nhưng e2e vẫn `Đang vận chuyển` → MV lag |
| `delivery_date_in_future` | `delivery_date_1` > hôm nay — đơn future (acceptable cho plan tới) |
| `outlier_delivery_date` | `delivery_date_1` < 2020 hoặc > 90 ngày tương lai → lỗi parse date |
| `drop_no_reason` | `e2e_label='Kế hoạch hủy'` nhưng `remark_2` rỗng → mất lý do drop |

In [45]:
df_biz = q("""
SELECT
  countIf(status = 'Cancel' AND san_luong_giao_cse > 0)
        AS cancel_but_delivered,
  countIf(e2e_label = 'Đã vận chuyển' AND ata_den IS NULL)
        AS delivered_label_but_no_ata,
  countIf(e2e_label = 'Đang vận chuyển' AND atd_den IS NULL)
        AS shipping_label_but_no_atd,
  countIf(e2e_label = 'Đã xuất kho'
          AND atd_den IS NOT NULL AND thoi_gian_di IS NOT NULL)
        AS shipped_label_but_has_full_stm,
  countIf(ata_den IS NOT NULL AND e2e_label = 'Đang vận chuyển')
        AS ata_set_but_still_shipping,
  countIf(toDate(delivery_date_1) > today())
        AS delivery_date_in_future,
  countIf(delivery_date_1 < toDateTime('2020-01-01')
          OR delivery_date_1 > now() + INTERVAL 90 DAY)
        AS outlier_delivery_date,
  countIf(e2e_label = 'Kế hoạch hủy' AND (remark_2 IS NULL OR remark_2 = ''))
        AS drop_no_reason,
  count() AS total_rows_in_window
FROM {T_MAIN}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
""")
total = int(df_biz['total_rows_in_window'].iloc[0])
out = df_biz.drop(columns=['total_rows_in_window']).T.rename(columns={0: 'violations'})
out['pct_window'] = (out['violations'] / total * 100).round(3) if total else 0
display(Markdown(f'**Business rule violations (window có {total:,} row)**'))
display(out)


**Business rule violations (window có 101,425 row)**

,violations,pct_window
cancel_but_delivered,0,0.00
delivered_label_but_no_ata,5,0.01
shipping_label_but_no_atd,0,0.00
shipped_label_but_has_full_stm,0,0.00
ata_set_but_still_shipping,0,0.00
delivery_date_in_future,0,0.00
outlier_delivery_date,0,0.00
drop_no_reason,199,0.20


## L7 · Bất thường — Key uniqueness & cross-MV parity

- **Dup key** — `(so, orderlinenumber)` phải unique trong window. Nếu dup → 1 đơn bị double-count khi sum volume.
- **Cross-MV parity** — `mv_flash_and_drop_report` = UNION ALL của `mv_flash_report` + `mv_dropped_report`. Tổng row combined phải = tổng 2 nguồn.

Diff parity > 0 → 1 trong 2 MV gốc đang lag refresh (vd `mv_flash_report` refresh 5' nhưng `mv_flash_and_drop_report` refresh 15').

In [46]:
df_dup = q("""
SELECT count() AS dup_key_pairs, coalesce(sum(c), 0) AS dup_row_total
FROM (
  SELECT so, orderlinenumber, count() AS c
  FROM {T_MAIN}
  WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
  GROUP BY so, orderlinenumber
  HAVING count() > 1
)
""")
display(Markdown('**Duplicate `(so, orderlinenumber)` trong window (kỳ vọng = 0)**'))
display(df_dup)

df_parity = q("""
SELECT
  (SELECT count() FROM {T_MAIN}
   WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')) AS rows_combined,
  (SELECT count() FROM {T_FLASH}
   WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')) AS rows_flash,
  (SELECT count() FROM {T_DROP}
   WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')) AS rows_dropped
""")
df_parity['flash_plus_dropped'] = df_parity['rows_flash'] + df_parity['rows_dropped']
df_parity['parity_diff'] = df_parity['flash_plus_dropped'] - df_parity['rows_combined']
display(Markdown('**Cross-MV parity — `combined = flash + dropped` (UNION ALL)**'))
display(df_parity.T.rename(columns={0: 'rows'}))


**Duplicate `(so, orderlinenumber)` trong window (kỳ vọng = 0)**

,dup_key_pairs,dup_row_total
0,0,0


**Cross-MV parity — `combined = flash + dropped` (UNION ALL)**

,rows
rows_combined,101425
rows_flash,101040
rows_dropped,385
flash_plus_dropped,101425
parity_diff,0


## L8 · Bất thường — Timestamp ordering

Quan hệ thời gian kỳ vọng:

`ngay_tao_don ≤ thoi_gian_gui_thau ≤ etd_chuyen_gui_thau ≤ eta_giao_hang_cho_npp`
`etd_chuyen_gui_thau ≤ atd_den ≤ ata_den`
`gio_ra_dock ≤ thoi_gian_di`
`atd_den - actual_ship_date < 1 day` (cùng phiên xuất)

Mọi `*_after_*` count > 0 → có data violate, ưu tiên drill `ata_before_etd` vì hỏng metric on-time.

In [47]:
df_ts = q("""
SELECT
  countIf(ngay_tao_don IS NOT NULL AND delivery_date_1 IS NOT NULL
          AND ngay_tao_don > delivery_date_1)
        AS create_after_delivery,
  countIf(thoi_gian_gui_thau IS NOT NULL AND etd_chuyen_gui_thau IS NOT NULL
          AND thoi_gian_gui_thau > etd_chuyen_gui_thau)
        AS bid_after_etd,
  countIf(etd_chuyen_gui_thau IS NOT NULL AND eta_giao_hang_cho_npp IS NOT NULL
          AND etd_chuyen_gui_thau > eta_giao_hang_cho_npp)
        AS etd_after_eta,
  countIf(atd_den IS NOT NULL AND ata_den IS NOT NULL
          AND atd_den > ata_den)
        AS atd_after_ata,
  countIf(ata_den IS NOT NULL AND etd_chuyen_gui_thau IS NOT NULL
          AND ata_den < etd_chuyen_gui_thau)
        AS ata_before_etd,
  countIf(actual_ship_date IS NOT NULL AND atd_den IS NOT NULL
          AND actual_ship_date > atd_den + INTERVAL 1 DAY)
        AS asd_far_after_atd,
  countIf(gio_ra_dock IS NOT NULL AND thoi_gian_di IS NOT NULL
          AND gio_ra_dock > thoi_gian_di)
        AS ra_dock_after_di,
  count() AS total_rows_in_window
FROM {T_MAIN}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
""")
total = int(df_ts['total_rows_in_window'].iloc[0])
out = df_ts.drop(columns=['total_rows_in_window']).T.rename(columns={0: 'violations'})
out['pct_window'] = (out['violations'] / total * 100).round(3) if total else 0
display(Markdown(f'**Timestamp ordering violations (window có {total:,} row)**'))
display(out)


**Timestamp ordering violations (window có 101,425 row)**

,violations,pct_window
create_after_delivery,749,0.74
bid_after_etd,385,0.38
etd_after_eta,0,0.00
atd_after_ata,1,0.00
ata_before_etd,395,0.39
asd_far_after_atd,1802,1.78
ra_dock_after_di,577,0.57


## Summary — chạy cuối, rollup L1-L8

Bảng dashboard gom **anomaly count L4-L8** + **key stats L1** trong 1 cell. Chạy sau khi đã review chi tiết các section trên — Summary là rollup cuối để biết "window này có cần action gì không".

**Status icon:**

- ✓ OK — pass threshold rule (vd `= 0` cho anomaly count)
- ✗ FAIL — vi phạm threshold → drill xuống section tương ứng (L4-L8 ở dưới) hoặc dùng L9 sample bad rows
- — (info) — chỉ số tham khảo, không có threshold (vd `distinct_so`, `delivery_future`)

> Summary chỉ tracking 21 metric chính. Để xem chi tiết per-dim (top-30 whseid / brand / customer), chạy L1-L8.


In [48]:
# Single query gom anomaly count L4-L8 + key stats L1
m = q("""
SELECT
  -- L1 health
  count()                                                              AS rows_mtd,
  uniq(so)                                                             AS distinct_so,
  toDateTime(max(ngay_tao_don), 'Asia/Ho_Chi_Minh')                    AS max_ngay_tao_don,
  dateDiff('minute', max(ngay_tao_don), now())                         AS lag_min_ngay_tao_don,
  -- L4 NULL critical
  countIf(so IS NULL OR so = '')                                       AS l4_so_null,
  countIf(whseid IS NULL OR whseid = '')                               AS l4_whseid_null,
  countIf(delivery_date_1 IS NULL)                                     AS l4_delivery_date_null,
  countIf(original_cse IS NULL OR original_cse = 0)                    AS l4_original_cse_zero,
  countIf(e2e_label IS NULL OR e2e_label = '')                         AS l4_e2e_label_null,
  -- L5 Volume integrity
  countIf(original_cse < 0)                                            AS l5_neg_original_cse,
  countIf(shipped_cse > original_cse AND original_cse > 0)             AS l5_shipped_gt_plan,
  countIf(san_luong_giao_cse > shipped_cse AND shipped_cse > 0)        AS l5_delivered_gt_shipped,
  -- L6 Business rule
  countIf(status = 'Cancel' AND san_luong_giao_cse > 0)                AS l6_cancel_but_delivered,
  countIf(e2e_label = 'Đã vận chuyển' AND ata_den IS NULL)             AS l6_delivered_no_ata,
  countIf(e2e_label = 'Kế hoạch hủy' AND (remark_2 IS NULL OR remark_2 = ''))
                                                                       AS l6_drop_no_reason,
  countIf(toDate(delivery_date_1) > today())                           AS l6_delivery_future,
  -- L8 Timestamp
  countIf(atd_den IS NOT NULL AND ata_den IS NOT NULL AND atd_den > ata_den)
                                                                       AS l8_atd_after_ata,
  countIf(ata_den IS NOT NULL AND etd_chuyen_gui_thau IS NOT NULL
          AND ata_den < etd_chuyen_gui_thau)                           AS l8_ata_before_etd,
  countIf(ngay_tao_don IS NOT NULL AND delivery_date_1 IS NOT NULL
          AND ngay_tao_don > delivery_date_1)                          AS l8_create_after_delivery
FROM {T_MAIN}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
""").iloc[0]

# L7 — duplicate keys + cross-MV parity (aggregation khác nên tách riêng)
dup = int(q("""
SELECT count() AS c FROM (
  SELECT so, orderlinenumber, count() AS c FROM {T_MAIN}
  WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
  GROUP BY so, orderlinenumber HAVING count() > 1
)
""")['c'].iloc[0])
par = q("""
SELECT
  (SELECT count() FROM {T_MAIN}
   WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')) AS combined,
  (SELECT count() FROM {T_FLASH}
   WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')) AS flash,
  (SELECT count() FROM {T_DROP}
   WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')) AS dropped
""").iloc[0]
parity_diff = int((par['flash'] + par['dropped']) - par['combined'])

# Status helpers
_OK, _FAIL, _INFO = '✓ OK', '✗ FAIL', '—'
def _eq0(v): return _OK if int(v) == 0 else _FAIL
def _gt0(v): return _OK if int(v) > 0 else _FAIL
def _lt(v, t): return _OK if int(v) < t else _FAIL

rows = [
    ('L1 Health',   'Rows MTD',                                int(m['rows_mtd']),                _gt0(m['rows_mtd']),                  '> 0'),
    ('L1 Health',   'Distinct SO',                             int(m['distinct_so']),             _INFO,                                'info'),
    ('L1 Health',   'Max ngay_tao_don (UTC+7)',                str(m['max_ngay_tao_don']),        _INFO,                                'info'),
    ('L1 Health',   'Lag ngay_tao_don (phút vs now)',          int(m['lag_min_ngay_tao_don']),    _lt(m['lag_min_ngay_tao_don'], 60),   '< 60 phút'),
    ('L4 NULL',     'so NULL/empty',                           int(m['l4_so_null']),              _eq0(m['l4_so_null']),                '= 0'),
    ('L4 NULL',     'whseid NULL/empty',                       int(m['l4_whseid_null']),          _eq0(m['l4_whseid_null']),            '= 0'),
    ('L4 NULL',     'delivery_date_1 NULL',                    int(m['l4_delivery_date_null']),   _eq0(m['l4_delivery_date_null']),     '= 0'),
    ('L4 NULL',     'original_cse = 0 / NULL',                 int(m['l4_original_cse_zero']),    _eq0(m['l4_original_cse_zero']),      '= 0'),
    ('L4 NULL',     'e2e_label NULL/empty',                    int(m['l4_e2e_label_null']),       _eq0(m['l4_e2e_label_null']),         '= 0'),
    ('L5 Volume',   'original_cse < 0',                        int(m['l5_neg_original_cse']),     _eq0(m['l5_neg_original_cse']),       '= 0'),
    ('L5 Volume',   'shipped_cse > original_cse',              int(m['l5_shipped_gt_plan']),      _eq0(m['l5_shipped_gt_plan']),        '= 0'),
    ('L5 Volume',   'delivered_cse > shipped_cse',             int(m['l5_delivered_gt_shipped']), _eq0(m['l5_delivered_gt_shipped']),   '= 0'),
    ('L6 Business', 'status=Cancel & đã giao',                 int(m['l6_cancel_but_delivered']), _eq0(m['l6_cancel_but_delivered']),   '= 0 (critical)'),
    ('L6 Business', 'e2e=Đã vận chuyển & ATA NULL (STM lag)',  int(m['l6_delivered_no_ata']),     _INFO,                                'info'),
    ('L6 Business', 'e2e=Kế hoạch hủy & remark_2 rỗng',        int(m['l6_drop_no_reason']),       _eq0(m['l6_drop_no_reason']),         '= 0'),
    ('L6 Business', 'delivery_date trong tương lai',           int(m['l6_delivery_future']),      _INFO,                                'info'),
    ('L7 Key',      'Duplicate (so, orderlinenumber)',         dup,                               _eq0(dup),                            '= 0'),
    ('L7 Key',      'Parity diff (combined − flash − dropped)', parity_diff,                      _eq0(parity_diff),                    '= 0'),
    ('L8 Time',     'atd_den > ata_den',                       int(m['l8_atd_after_ata']),        _eq0(m['l8_atd_after_ata']),          '= 0'),
    ('L8 Time',     'ata_den < etd_chuyen_gui_thau',           int(m['l8_ata_before_etd']),       _eq0(m['l8_ata_before_etd']),         '= 0'),
    ('L8 Time',     'ngay_tao_don > delivery_date_1',          int(m['l8_create_after_delivery']), _eq0(m['l8_create_after_delivery']), '= 0'),
]
df_summary = pd.DataFrame(rows, columns=['Section', 'Metric', 'Value', 'Status', 'Rule'])

def _color_status(s):
    out = []
    for v in s:
        if v == _OK:
            out.append('background-color: #DCFCE7; color: #166534; font-weight: 600')
        elif v == _FAIL:
            out.append('background-color: #FEE2E2; color: #991B1B; font-weight: 600')
        else:
            out.append('color: #6B7280')
    return out

n_ok   = int((df_summary['Status'] == _OK).sum())
n_fail = int((df_summary['Status'] == _FAIL).sum())
n_info = int((df_summary['Status'] == _INFO).sum())

# Ẩn dòng anomaly có Value == 0 (pass threshold) — chỉ giữ dòng đáng theo dõi.
# L1 Health luôn show (stat overview, không phải anomaly).
def _is_anomaly_zero(row):
    if row['Section'] == 'L1 Health':
        return False
    try:
        return int(row['Value']) == 0
    except (TypeError, ValueError):
        return False

mask_visible = ~df_summary.apply(_is_anomaly_zero, axis=1)
df_visible = df_summary[mask_visible]
n_hidden = len(df_summary) - len(df_visible)

display(Markdown(
    f"**Window:** `{PARAMS['from_date']}` → `{PARAMS['to_date']}`  ·  "
    f"**Date col:** `{PARAMS['date_col']}` ({PARAMS['date_label']})  ·  "
    f"**UOM:** `{PARAMS['uom_col']}`\n\n"
    f"**Result:** ✓ OK = {n_ok}  ·  ✗ FAIL = {n_fail}  ·  — info = {n_info}"
    + (f"  →  ⚠ **{n_fail} metric vi phạm — drill bằng L9/L10**" if n_fail else "  →  ✓ Data healthy, không có anomaly")
    + (f"\n\n_({n_hidden} dòng anomaly = 0 đã ẩn — set `SHOW_ALL = True` để xem hết)_" if n_hidden else "")
))

SHOW_ALL = False  # đổi True để xem toàn bộ 21 metric kể cả dòng = 0
display((df_summary if SHOW_ALL else df_visible)
        .style.apply(_color_status, subset=['Status']).hide(axis='index'))


**Window:** `2026-05-01` → `2026-05-28`  ·  **Date col:** `delivery_date_1` (Ngày GI)  ·  **UOM:** `original_cse`

**Result:** ✓ OK = 11  ·  ✗ FAIL = 6  ·  — info = 4  →  ⚠ **6 metric vi phạm — drill bằng L9/L10**

_(11 dòng anomaly = 0 đã ẩn — set `SHOW_ALL = True` để xem hết)_

Section,Metric,Value,Status,Rule
L1 Health,Rows MTD,101425,✓ OK,> 0
L1 Health,Distinct SO,15798,—,info
L1 Health,Max ngay_tao_don (UTC+7),2026-05-28 11:36:50+07:00,—,info
L1 Health,Lag ngay_tao_don (phút vs now),194,✗ FAIL,< 60 phút
L5 Volume,delivered_cse > shipped_cse,1323,✗ FAIL,= 0
L6 Business,e2e=Đã vận chuyển & ATA NULL (STM lag),5,—,info
L6 Business,e2e=Kế hoạch hủy & remark_2 rỗng,199,✗ FAIL,= 0
L8 Time,atd_den > ata_den,1,✗ FAIL,= 0
L8 Time,ata_den < etd_chuyen_gui_thau,395,✗ FAIL,= 0
L8 Time,ngay_tao_don > delivery_date_1,749,✗ FAIL,= 0


## L9 · Drill listout — đơn / chuyến vi phạm

Mỗi sub-section dưới đây drill 1 loại bất thường thành **danh sách đơn cụ thể** (≤ 100 row, sort theo độ nghiêm trọng giảm dần). Dùng để:

- Copy `so` / `orderlinenumber` paste vào IT khách / `/da-ch` để mở case.
- Filter `whseid` / `khu_vuc_doi_xe` / `customer_code` để biết hotspot.
- Đánh giá xem violation là **systematic bug** (lặp ở 1 whseid) hay **edge case** (rải rác).

| Sub | Đối tượng | Threshold | Sort key |
|---|---|---|---|
| L9.1 | L6 — `e2e = Đã vận chuyển` & `ata_den` NULL (STM lag) | info | gi_date DESC |
| L9.2 | L5 — Delivered > Shipped (STM ghi sai) | = 0 | overdelivery_cse DESC |
| L9.3 | L6 — `status = Cancel` nhưng đã giao | = 0 (critical) | delivered_cse DESC |
| L9.4 | L7 — Duplicate `(so, orderlinenumber)` | = 0 | dup_count DESC |
| **L9.5** | **L8 — `ata_den < etd_chuyen_gui_thau` (ATA về trước ETD)** | **= 0** | **hours_early DESC** |
| L9.6 | L8 — `atd_den > ata_den` (STM ghi ngược) | = 0 | hours_swapped DESC |
| L9.7 | L8 — `ngay_tao_don > delivery_date_1` (tạo đơn sau giao) | = 0 | days_late_create DESC |
| L9.8 | L8 — `bid_after_etd` / `etd_after_eta` (planning sai chuỗi) | = 0 | gi_date DESC |
| L9.9 | L8 — `gio_ra_dock > thoi_gian_di` (ra dock sau khi đi) | = 0 | hours_swapped DESC |

> Mỗi cell tự rerun độc lập — dùng PARAMS đã set ở L0. Đổi WHERE để filter thêm theo whseid/customer khi cần.

### L9.1 · STM lag — đơn `Đã vận chuyển` mà `ata_den` chưa có

Đơn có `e2e_label = 'Đã vận chuyển'` nhưng STM chưa ghi `ata_den`. Đây là dấu hiệu **STM lag** (telematics chưa kịp đóng chuyến) hơn là bug — show để monitor backlog. Nếu cùng đơn lag > 24h → escalate cho IT khách.

In [49]:
# L9.1 — STM lag: e2e_label = 'Đã vận chuyển' nhưng ata_den IS NULL
q("""
SELECT so, orderlinenumber, whseid, customer_code, customer_name,
       status, e2e_label, khu_vuc_doi_xe,
       toDateTime(delivery_date_1, 'Asia/Ho_Chi_Minh')      AS gi_date,
       toDateTime(actual_ship_date, 'Asia/Ho_Chi_Minh')     AS actual_ship,
       toDateTime(atd_den, 'Asia/Ho_Chi_Minh')              AS atd,
       toDateTime(ata_den, 'Asia/Ho_Chi_Minh')              AS ata,
       original_cse, shipped_cse, san_luong_giao_cse,
       remark_2
FROM {T_MAIN}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
  AND e2e_label = 'Đã vận chuyển'
  AND ata_den IS NULL
ORDER BY delivery_date_1 DESC, so
LIMIT 100
""")


,so,orderlinenumber,whseid,customer_code,customer_name,status,e2e_label,khu_vuc_doi_xe,gi_date,actual_ship,atd,ata,original_cse,shipped_cse,san_luong_giao_cse,remark_2
0,8482501532,00007,NKD,6840063,Cong Ty TNHH KD va DV Hien Duong,ShipCompleted,Đã vận chuyển,North East - North West,2026-05-11 14:00:00+07:00,2026-05-11 11:26:48+07:00,2026-05-11 06:59:41+07:00,NaT,14.00,14.00,4.00,
1,8482501533,00001,NKD,6840079,Cong ty TNHH TM va DV Dung Tien,ShipCompleted,Đã vận chuyển,North East - North West,2026-05-11 14:00:00+07:00,2026-05-11 11:43:51+07:00,2026-05-11 10:03:26+07:00,NaT,10.00,10.00,3.00,
2,8482501593,00001,NKD,6840116,Cong Ty TNHH Thuong Mai D&T Vinh Ph,ShipCompleted,Đã vận chuyển,North East - North West,2026-05-11 14:00:00+07:00,2026-05-11 10:05:30+07:00,2026-05-11 09:16:02+07:00,NaT,8.00,8.00,3.00,
3,8482501593,00007,NKD,6840116,Cong Ty TNHH Thuong Mai D&T Vinh Ph,ShipCompleted,Đã vận chuyển,North East - North West,2026-05-11 14:00:00+07:00,2026-05-11 10:05:30+07:00,2026-05-11 09:16:02+07:00,NaT,14.00,14.00,4.00,
4,8482501594,00001,NKD,7093363,Cong Ty CP KD TM Hoang Ha,ShipCompleted,Đã vận chuyển,North East - North West,2026-05-11 14:00:00+07:00,2026-05-11 10:05:48+07:00,2026-05-11 09:16:02+07:00,NaT,8.00,8.00,3.00,


### L9.2 · Volume — Delivered > Shipped

`san_luong_giao_cse > shipped_cse` → STM báo giao nhiều hơn WMS báo xuất kho. Lý do thường gặp: (1) STM ghi sai số CSE/case, (2) WMS delete-lag (đã xoá `shipped` nhưng `delivered` chưa cập nhật), (3) đơn bị tách-gộp sai. Sort theo `overdelivery_cse DESC` để xem case nặng nhất.

In [50]:
# L9.2 — Delivered > Shipped
q("""
SELECT so, orderlinenumber, whseid, customer_code, customer_name,
       status, e2e_label, khu_vuc_doi_xe,
       toDateTime(delivery_date_1, 'Asia/Ho_Chi_Minh')      AS gi_date,
       original_cse, shipped_cse, san_luong_giao_cse,
       round(san_luong_giao_cse - shipped_cse, 2)           AS overdelivery_cse,
       round((san_luong_giao_cse - shipped_cse)
             / nullIf(shipped_cse, 0) * 100, 2)             AS overdelivery_pct
FROM {T_MAIN}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
  AND san_luong_giao_cse > shipped_cse
  AND shipped_cse > 0
ORDER BY overdelivery_cse DESC
LIMIT 100
""")


,so,orderlinenumber,whseid,customer_code,customer_name,status,e2e_label,khu_vuc_doi_xe,gi_date,original_cse,shipped_cse,san_luong_giao_cse,overdelivery_cse,overdelivery_pct
0,8482512381,00002,BKD1,7126336,Cong Ty TNHH Mot Thanh Vien ThuongVa Phan Phoi Nghia Phat,ShipCompleted,Đã vận chuyển,Ho Chi Minh,2026-05-20 14:00:00+07:00,178.00,178.00,690.00,512.00,287.64
1,8482518590,00011,BKD1,7107116,Cong Ty TNHH MTV TM DV Tung Duong N,ShipCompleted,Đã vận chuyển,South East,2026-05-26 14:00:00+07:00,11.00,11.00,440.00,429.00,"3,900.00"
2,8482518931,00032,NKD,7080372,CN Cong ty TNHH Phan Phoi Tien Tien,ShipCompleted,Đã vận chuyển,Ha Noi,2026-05-27 14:00:00+07:00,28.00,28.00,400.00,372.00,"1,328.57"
3,8482512704,00002,BKD1,7121034,Cong Ty TNHH MTV SXTMDV Vinh Hung T,ShipCompleted,Đã vận chuyển,Ho Chi Minh,2026-05-20 14:00:00+07:00,55.00,55.00,373.00,318.00,578.18
4,8482492803,00014,BKD1,7125012,Cong Ty TNHH Thuong Mai Mai PhuongBa Ria - Vung Tau,ShipCompleted,Đã vận chuyển,South East,2026-05-04 14:00:00+07:00,26.00,26.00,330.00,304.00,"1,169.23"
5,8482495222,00002,BKD1,7121034,Cong Ty TNHH MTV SXTMDV Vinh Hung T,ShipCompleted,Đã vận chuyển,Ho Chi Minh,2026-05-05 14:00:00+07:00,55.00,55.00,330.00,275.00,500.00
6,8482515369,00018,NKD,7115415,Cong Ty TNHH Hung Minh Phat,ShipCompleted,Đã vận chuyển,North Central Coast,2026-05-21 14:00:00+07:00,1.00,1.00,216.00,215.00,"21,500.00"
7,8482510269,00006,BKD1,7157359,Cong Ty TNHH TM DV San Xuat Boi Boi,ShipCompleted,Đã vận chuyển,Ho Chi Minh,2026-05-19 14:00:00+07:00,23.00,23.00,220.00,197.00,856.52
8,8482518931,00034,NKD,7080372,CN Cong ty TNHH Phan Phoi Tien Tien,ShipCompleted,Đã vận chuyển,Ha Noi,2026-05-27 14:00:00+07:00,10.00,10.00,200.00,190.00,"1,900.00"
9,8482518931,00033,NKD,7080372,CN Cong ty TNHH Phan Phoi Tien Tien,ShipCompleted,Đã vận chuyển,Ha Noi,2026-05-27 14:00:00+07:00,18.00,18.00,202.00,184.00,"1,022.22"


### L9.3 · Business — `status = Cancel` nhưng đã giao

**Critical anomaly**. Đơn được đánh dấu Cancel trên TMS nhưng vẫn có `san_luong_giao_cse > 0` — kế toán/billing sẽ tính sai. Mọi case xuất hiện ở đây nên raise ngay cho operations team.

In [51]:
# L9.3 — status=Cancel nhưng đã giao
q("""
SELECT so, orderlinenumber, whseid, customer_code, customer_name,
       status, e2e_label, khu_vuc_doi_xe,
       toDateTime(delivery_date_1, 'Asia/Ho_Chi_Minh')      AS gi_date,
       toDateTime(ata_den, 'Asia/Ho_Chi_Minh')              AS ata,
       original_cse, shipped_cse, san_luong_giao_cse,
       remark_2
FROM {T_MAIN}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
  AND status = 'Cancel'
  AND san_luong_giao_cse > 0
ORDER BY san_luong_giao_cse DESC
LIMIT 100
""")


""


### L9.4 · Key — Duplicate `(so, orderlinenumber)`

Cùng cặp `(so, orderlinenumber)` xuất hiện > 1 lần trong window. Bug trong pipeline — 1 đơn double-count khi sum volume. `groupArray` giúp xem các giá trị `whseid` / `e2e_label` khác nhau của các bản sao để xác định nguyên nhân (vd cùng SO ở 2 kho hoặc 2 trạng thái).

In [52]:
# L9.4 — Duplicate (so, orderlinenumber)
q("""
SELECT so, orderlinenumber,
       count()                                      AS dup_count,
       groupArray(coalesce(whseid, ''))             AS whseids,
       groupArray(coalesce(e2e_label, ''))          AS e2e_labels,
       groupArray(coalesce(status, ''))             AS statuses,
       round(sum(original_cse), 2)                  AS sum_original_cse,
       round(sum(san_luong_giao_cse), 2)            AS sum_delivered_cse
FROM {T_MAIN}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
GROUP BY so, orderlinenumber
HAVING count() > 1
ORDER BY dup_count DESC, sum_original_cse DESC
LIMIT 100
""")


""


### L9.5 · L8 timestamp — `ata_den < etd_chuyen_gui_thau` (ATA về trước ETD)

**Hỏng on-time metric** — nếu ATA về trước ETD thì on-time auto pass mà thực ra plan đang sai. Có 2 nguyên nhân thường gặp:

1. **ETD bị fix tay sau khi giao** (đẩy plan về tương lai) — STM giữ ATA cũ, nhưng ETD bị update muộn.
2. **STM ghi nhầm `ata_den`** (lấy timestamp từ chuyến khác).

`hours_early` = ETD − ATA (đơn vị: giờ). Sort DESC để xem case lệch nặng nhất.

**Bảng 1** — Listing chi tiết per đơn (100 row nặng nhất).
**Bảng 2** — Hotspot: nhóm theo `whseid` × `khu_vuc_doi_xe` để biết kho/vùng nào lặp lại — pattern lặp = systematic bug, không phải edge case.

In [53]:
# L9.5a — Listing đơn ata_before_etd, sort hours_early DESC
df_ata_before_etd = q("""
SELECT so, orderlinenumber, whseid, customer_code, customer_name,
       status, e2e_label, khu_vuc_doi_xe, group_name AS kenh,
       toDateTime(delivery_date_1, 'Asia/Ho_Chi_Minh')      AS gi_date,
       toDateTime(etd_chuyen_gui_thau, 'Asia/Ho_Chi_Minh')  AS etd,
       toDateTime(atd_den, 'Asia/Ho_Chi_Minh')              AS atd,
       toDateTime(ata_den, 'Asia/Ho_Chi_Minh')              AS ata,
       round(dateDiff('minute', ata_den, etd_chuyen_gui_thau) / 60.0, 2)
                                                            AS hours_early,
       round(dateDiff('day',    ata_den, etd_chuyen_gui_thau), 0)
                                                            AS days_early,
       original_cse, shipped_cse, san_luong_giao_cse
FROM {T_MAIN}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
  AND ata_den IS NOT NULL
  AND etd_chuyen_gui_thau IS NOT NULL
  AND ata_den < etd_chuyen_gui_thau
ORDER BY hours_early DESC
LIMIT 100
""")
display(Markdown(f'**L9.5a · Bảng 1 — Listing chi tiết `ata_before_etd` (top 100 theo `hours_early`)**  ·  '
                 f'total found = `{len(df_ata_before_etd)}` row trong top 100'))
display(df_ata_before_etd)


**L9.5a · Bảng 1 — Listing chi tiết `ata_before_etd` (top 100 theo `hours_early`)**  ·  total found = `100` row trong top 100

,so,orderlinenumber,whseid,customer_code,customer_name,status,e2e_label,khu_vuc_doi_xe,kenh,gi_date,etd,atd,ata,hours_early,days_early,original_cse,shipped_cse,san_luong_giao_cse
0,8482507941,00006,BKD1,7121034,Cong Ty TNHH MTV SXTMDV Vinh Hung T,ShipCompleted,Đã vận chuyển,Ho Chi Minh,GT,2026-05-15 14:00:00+07:00,2026-05-15 20:00:00+07:00,2026-05-14 19:33:36+07:00,2026-05-14 21:29:31+07:00,22.52,1,253.00,253.00,38.00
1,8482507941,00002,BKD1,7121034,Cong Ty TNHH MTV SXTMDV Vinh Hung T,ShipCompleted,Đã vận chuyển,Ho Chi Minh,GT,2026-05-15 14:00:00+07:00,2026-05-15 20:00:00+07:00,2026-05-14 19:33:36+07:00,2026-05-14 21:29:31+07:00,22.52,1,100.00,100.00,100.00
2,8482507941,00003,BKD1,7121034,Cong Ty TNHH MTV SXTMDV Vinh Hung T,ShipCompleted,Đã vận chuyển,Ho Chi Minh,GT,2026-05-15 14:00:00+07:00,2026-05-15 20:00:00+07:00,2026-05-14 19:33:36+07:00,2026-05-14 21:29:31+07:00,22.52,1,50.00,50.00,50.00
3,8482507941,00005,BKD1,7121034,Cong Ty TNHH MTV SXTMDV Vinh Hung T,ShipCompleted,Đã vận chuyển,Ho Chi Minh,GT,2026-05-15 14:00:00+07:00,2026-05-15 20:00:00+07:00,2026-05-14 19:33:36+07:00,2026-05-14 21:29:31+07:00,22.52,1,6.00,6.00,6.00
4,8482507941,00004,BKD1,7121034,Cong Ty TNHH MTV SXTMDV Vinh Hung T,ShipCompleted,Đã vận chuyển,Ho Chi Minh,GT,2026-05-15 14:00:00+07:00,2026-05-15 20:00:00+07:00,2026-05-14 19:33:36+07:00,2026-05-14 21:29:31+07:00,22.52,1,88.00,88.00,88.00
5,8482507941,00001,BKD1,7121034,Cong Ty TNHH MTV SXTMDV Vinh Hung T,ShipCompleted,Đã vận chuyển,Ho Chi Minh,GT,2026-05-15 14:00:00+07:00,2026-05-15 20:00:00+07:00,2026-05-14 19:33:36+07:00,2026-05-14 21:29:31+07:00,22.52,1,150.00,150.00,150.00
6,8482507940,00002,BKD1,7121034,Cong Ty TNHH MTV SXTMDV Vinh Hung T,ShipCompleted,Đã vận chuyển,Ho Chi Minh,GT,2026-05-15 14:00:00+07:00,2026-05-15 20:00:00+07:00,2026-05-14 18:41:46+07:00,2026-05-14 21:32:10+07:00,22.47,1,26.00,26.00,26.00
7,8482507940,00003,BKD1,7121034,Cong Ty TNHH MTV SXTMDV Vinh Hung T,ShipCompleted,Đã vận chuyển,Ho Chi Minh,GT,2026-05-15 14:00:00+07:00,2026-05-15 20:00:00+07:00,2026-05-14 18:41:46+07:00,2026-05-14 21:32:10+07:00,22.47,1,48.00,48.00,48.00
8,8482507940,00004,BKD1,7121034,Cong Ty TNHH MTV SXTMDV Vinh Hung T,ShipCompleted,Đã vận chuyển,Ho Chi Minh,GT,2026-05-15 14:00:00+07:00,2026-05-15 20:00:00+07:00,2026-05-14 18:41:46+07:00,2026-05-14 21:32:10+07:00,22.47,1,36.00,36.00,36.00
9,8482507940,00007,BKD1,7121034,Cong Ty TNHH MTV SXTMDV Vinh Hung T,ShipCompleted,Đã vận chuyển,Ho Chi Minh,GT,2026-05-15 14:00:00+07:00,2026-05-15 20:00:00+07:00,2026-05-14 18:41:46+07:00,2026-05-14 21:32:10+07:00,22.47,1,52.00,52.00,52.00


In [54]:
# L9.5b — Hotspot ata_before_etd: nhóm theo whseid × khu_vuc_doi_xe
df_ata_hotspot = q("""
SELECT coalesce(nullIf(whseid, ''), '(NULL)')                AS whseid,
       coalesce(nullIf(khu_vuc_doi_xe, ''), '(NULL)')        AS khu_vuc,
       count()                                               AS violations,
       uniq(so)                                              AS distinct_so,
       round(avg(dateDiff('minute', ata_den, etd_chuyen_gui_thau) / 60.0), 2)
                                                             AS avg_hours_early,
       round(max(dateDiff('minute', ata_den, etd_chuyen_gui_thau) / 60.0), 2)
                                                             AS max_hours_early,
       round(sum(original_cse), 2)                           AS sum_original_cse
FROM {T_MAIN}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
  AND ata_den IS NOT NULL
  AND etd_chuyen_gui_thau IS NOT NULL
  AND ata_den < etd_chuyen_gui_thau
GROUP BY whseid, khu_vuc
ORDER BY violations DESC
LIMIT 30
""")
display(Markdown('**L9.5b · Bảng 2 — Hotspot `ata_before_etd` theo `whseid × khu_vuc_doi_xe` (top 30)**'))
display(df_ata_hotspot)

# L9.5c — Hotspot theo customer (top 20)
df_ata_cust = q("""
SELECT coalesce(nullIf(customer_code, ''), '(NULL)')         AS customer_code,
       any(customer_name)                                    AS customer_name,
       count()                                               AS violations,
       uniq(so)                                              AS distinct_so,
       round(avg(dateDiff('minute', ata_den, etd_chuyen_gui_thau) / 60.0), 2)
                                                             AS avg_hours_early
FROM {T_MAIN}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
  AND ata_den IS NOT NULL
  AND etd_chuyen_gui_thau IS NOT NULL
  AND ata_den < etd_chuyen_gui_thau
GROUP BY customer_code
ORDER BY violations DESC
LIMIT 20
""")
display(Markdown('**L9.5c · Bảng 3 — Hotspot `ata_before_etd` theo `customer_code` (top 20)**'))
display(df_ata_cust)


**L9.5b · Bảng 2 — Hotspot `ata_before_etd` theo `whseid × khu_vuc_doi_xe` (top 30)**

,whseid,khu_vuc,violations,distinct_so,avg_hours_early,max_hours_early,sum_original_cse
0,BKD1,South East,127,15,1.02,2.27,"4,555.00"
1,BKD1,Ho Chi Minh,122,15,6.01,22.52,"3,577.00"
2,NKD,Ha Noi,84,9,1.72,2.58,539.00
3,NKD,North East - North West,31,4,0.28,0.47,218.56
4,BKD1,Mekong 2,21,4,0.23,0.35,228.00
5,BKD1,(NULL),10,3,0.23,0.65,"1,116.00"


**L9.5c · Bảng 3 — Hotspot `ata_before_etd` theo `customer_code` (top 20)**

,customer_code,customer_name,violations,distinct_so,avg_hours_early
0,7105620,BHX_HCM - Kho DC Vinh Loc,60,5,2.07
1,6840436,Trung Tam Phan Phoi Song Than,45,7,1.56
2,6840283,Cong Ty TNHH Aeon Viet Nam - CN BinDuong,37,4,0.38
3,7107116,Cong Ty TNHH MTV TM DV Tung Duong N,37,3,0.86
4,7121034,Cong Ty TNHH MTV SXTMDV Vinh Hung T,31,5,18.13
5,7160022,Cong Ty TNHH Anh Minh Chinh,26,4,0.68
6,6840276,Cong Ty CP Trung Tam Thuong Mai LotViet Nam - CN Ba Dinh,23,1,2.58
7,6840304,Cong Ty TNHH Banh Keo Thanh Loi,21,4,0.23
8,6840066,Cong Ty TNHH Thuong Mai Tong hopLong Hai,19,1,2.12
9,6840288,Cong Ty CP Trung Tam Thuong Mai LotViet Nam - CN Tan Binh,17,1,0.30


### L9.6 · L8 timestamp — `atd_den > ata_den` (ATD sau ATA — STM ghi ngược)

ATD (giờ rời điểm đầu) phải ≤ ATA (giờ đến điểm cuối). Vi phạm = STM ghi nhầm thứ tự event hoặc fix tay sai field. `hours_swapped` = ATD − ATA.

In [55]:
# L9.6 — atd_den > ata_den
q("""
SELECT so, orderlinenumber, whseid, customer_code, customer_name,
       status, e2e_label, khu_vuc_doi_xe,
       toDateTime(delivery_date_1, 'Asia/Ho_Chi_Minh')      AS gi_date,
       toDateTime(atd_den, 'Asia/Ho_Chi_Minh')              AS atd,
       toDateTime(ata_den, 'Asia/Ho_Chi_Minh')              AS ata,
       round(dateDiff('minute', ata_den, atd_den) / 60.0, 2) AS hours_swapped,
       original_cse, san_luong_giao_cse
FROM {T_MAIN}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
  AND atd_den IS NOT NULL AND ata_den IS NOT NULL
  AND atd_den > ata_den
ORDER BY hours_swapped DESC
LIMIT 100
""")


,so,orderlinenumber,whseid,customer_code,customer_name,status,e2e_label,khu_vuc_doi_xe,gi_date,atd,ata,hours_swapped,original_cse,san_luong_giao_cse
0,8482515207,00001,BKD1,7126336,Cong Ty TNHH Mot Thanh Vien ThuongVa Phan Phoi Nghia Phat,ShipCompleted,Đã vận chuyển,Ho Chi Minh,2026-05-21 14:00:00+07:00,2026-05-21 23:53:54+07:00,2026-05-21 23:00:00+07:00,0.88,15.00,15.00


### L9.7 · L8 timestamp — `ngay_tao_don > delivery_date_1` (tạo đơn sau ngày giao)

Đơn được tạo trên TMS SAU ngày `delivery_date_1` (GI). Vi phạm = master data hậu kỳ — đơn nhập tay vào TMS sau khi giao thực tế xong (vd reconcile cuối tháng). `days_late_create` = ngày tạo − ngày GI.

In [56]:
# L9.7 — ngay_tao_don > delivery_date_1
q("""
SELECT so, orderlinenumber, whseid, customer_code, customer_name,
       status, e2e_label, khu_vuc_doi_xe,
       toDateTime(ngay_tao_don,    'Asia/Ho_Chi_Minh')      AS created_at,
       toDateTime(delivery_date_1, 'Asia/Ho_Chi_Minh')      AS gi_date,
       dateDiff('day', delivery_date_1, ngay_tao_don)       AS days_late_create,
       original_cse, san_luong_giao_cse
FROM {T_MAIN}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
  AND ngay_tao_don IS NOT NULL AND delivery_date_1 IS NOT NULL
  AND ngay_tao_don > delivery_date_1
ORDER BY days_late_create DESC
LIMIT 100
""")


,so,orderlinenumber,whseid,customer_code,customer_name,status,e2e_label,khu_vuc_doi_xe,created_at,gi_date,days_late_create,original_cse,san_luong_giao_cse
0,8482510138,00033,NKD,7151387,Cong Ty Trach Nhiem Huu Han Dich VuThuong Mai Tong Hop Hoang Dat,ShipCompleted,Đã vận chuyển,North Central Coast,2026-05-18 23:01:55+07:00,2026-05-16 14:00:00+07:00,2,15.00,15.00
1,8482510138,00022,NKD,7151387,Cong Ty Trach Nhiem Huu Han Dich VuThuong Mai Tong Hop Hoang Dat,ShipCompleted,Đã vận chuyển,North Central Coast,2026-05-18 23:01:55+07:00,2026-05-16 14:00:00+07:00,2,1.00,1.00
2,8482510138,00011,NKD,7151387,Cong Ty Trach Nhiem Huu Han Dich VuThuong Mai Tong Hop Hoang Dat,ShipCompleted,Đã vận chuyển,North Central Coast,2026-05-18 23:01:55+07:00,2026-05-16 14:00:00+07:00,2,4.00,4.00
3,8482518623,00002,NKD,6840068,Cong Ty TNHH TM Anh Hue,ShipCompleted,Đã vận chuyển,North Central Coast,2026-05-25 23:01:55+07:00,2026-05-23 14:00:00+07:00,2,16.00,16.00
4,8482518602,00015,NKD,6840068,Cong Ty TNHH TM Anh Hue,ShipCompleted,Đã vận chuyển,North Central Coast,2026-05-25 23:01:47+07:00,2026-05-23 14:00:00+07:00,2,3.00,1.00
5,8482518602,00014,NKD,6840068,Cong Ty TNHH TM Anh Hue,ShipCompleted,Đã vận chuyển,North Central Coast,2026-05-25 23:01:47+07:00,2026-05-23 14:00:00+07:00,2,1.00,19.00
6,8482518602,00013,NKD,6840068,Cong Ty TNHH TM Anh Hue,ShipCompleted,Đã vận chuyển,North Central Coast,2026-05-25 23:01:47+07:00,2026-05-23 14:00:00+07:00,2,19.00,112.00
7,8482518602,00010,NKD,6840068,Cong Ty TNHH TM Anh Hue,ShipCompleted,Đã vận chuyển,North Central Coast,2026-05-25 23:01:47+07:00,2026-05-23 14:00:00+07:00,2,1.00,12.00
8,8482518602,00009,NKD,6840068,Cong Ty TNHH TM Anh Hue,ShipCompleted,Đã vận chuyển,North Central Coast,2026-05-25 23:01:47+07:00,2026-05-23 14:00:00+07:00,2,12.00,6.00
9,8482518602,00008,NKD,6840068,Cong Ty TNHH TM Anh Hue,ShipCompleted,Đã xuất kho,North Central Coast,2026-05-25 23:01:47+07:00,2026-05-23 14:00:00+07:00,2,6.00,0.00


### L9.8 · L8 timestamp — Planning sai chuỗi (`bid_after_etd` / `etd_after_eta`)

Hai vi phạm cùng cụm "kế hoạch ngược":

- `bid_after_etd`: gửi thầu (`thoi_gian_gui_thau`) sau ETD chuyến — bid hậu kỳ.
- `etd_after_eta`: ETD sau ETA — kế hoạch ngược thời gian.

`hours_gap` = giờ chênh, sort DESC.

In [57]:
# L9.8a — bid_after_etd (gửi thầu sau ETD)
df_bid = q("""
SELECT so, orderlinenumber, whseid, customer_code, status, e2e_label,
       toDateTime(delivery_date_1, 'Asia/Ho_Chi_Minh')      AS gi_date,
       toDateTime(thoi_gian_gui_thau, 'Asia/Ho_Chi_Minh')   AS bid_time,
       toDateTime(etd_chuyen_gui_thau, 'Asia/Ho_Chi_Minh')  AS etd,
       round(dateDiff('minute', etd_chuyen_gui_thau, thoi_gian_gui_thau) / 60.0, 2)
                                                            AS hours_gap,
       original_cse
FROM {T_MAIN}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
  AND thoi_gian_gui_thau IS NOT NULL AND etd_chuyen_gui_thau IS NOT NULL
  AND thoi_gian_gui_thau > etd_chuyen_gui_thau
ORDER BY hours_gap DESC
LIMIT 50
""")
display(Markdown('**L9.8a · `bid_after_etd` (top 50)**'))
display(df_bid)

# L9.8b — etd_after_eta (ETD sau ETA)
df_etd = q("""
SELECT so, orderlinenumber, whseid, customer_code, status, e2e_label,
       toDateTime(delivery_date_1, 'Asia/Ho_Chi_Minh')          AS gi_date,
       toDateTime(etd_chuyen_gui_thau, 'Asia/Ho_Chi_Minh')      AS etd,
       toDateTime(eta_giao_hang_cho_npp, 'Asia/Ho_Chi_Minh')    AS eta,
       round(dateDiff('minute', eta_giao_hang_cho_npp, etd_chuyen_gui_thau) / 60.0, 2)
                                                                AS hours_gap,
       original_cse
FROM {T_MAIN}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
  AND etd_chuyen_gui_thau IS NOT NULL AND eta_giao_hang_cho_npp IS NOT NULL
  AND etd_chuyen_gui_thau > eta_giao_hang_cho_npp
ORDER BY hours_gap DESC
LIMIT 50
""")
display(Markdown('**L9.8b · `etd_after_eta` (top 50)**'))
display(df_etd)


**L9.8a · `bid_after_etd` (top 50)**

,so,orderlinenumber,whseid,customer_code,status,e2e_label,gi_date,bid_time,etd,hours_gap,original_cse
0,8482511104,00001,BKD1,7123617,ShipCompleted,Đã xuất kho,2026-05-22 14:00:00+07:00,2026-05-21 22:05:29+07:00,2026-05-21 17:00:00+07:00,5.08,1.00
1,8482511175,00001,BKD1,7123143,ShipCompleted,Đã xuất kho,2026-05-22 14:00:00+07:00,2026-05-21 22:05:29+07:00,2026-05-21 17:00:00+07:00,5.08,1.00
2,8482510925,00002,BKD1,7124030,ShipCompleted,Đã xuất kho,2026-05-22 14:00:00+07:00,2026-05-21 22:05:29+07:00,2026-05-21 17:00:00+07:00,5.08,1.00
3,8482510583,00002,BKD1,7123143,ShipCompleted,Đã xuất kho,2026-05-22 14:00:00+07:00,2026-05-21 22:05:29+07:00,2026-05-21 17:00:00+07:00,5.08,5.00
4,8482510583,00003,BKD1,7123143,ShipCompleted,Đã xuất kho,2026-05-22 14:00:00+07:00,2026-05-21 22:05:29+07:00,2026-05-21 17:00:00+07:00,5.08,2.00
5,8482510583,00004,BKD1,7123143,ShipCompleted,Đã xuất kho,2026-05-22 14:00:00+07:00,2026-05-21 22:05:29+07:00,2026-05-21 17:00:00+07:00,5.08,2.00
6,8482511323,00003,BKD1,7124009,ShipCompleted,Đã xuất kho,2026-05-22 14:00:00+07:00,2026-05-21 22:05:29+07:00,2026-05-21 17:00:00+07:00,5.08,1.00
7,8482511149,00001,BKD1,7123607,ShipCompleted,Đã xuất kho,2026-05-22 14:00:00+07:00,2026-05-21 22:05:29+07:00,2026-05-21 17:00:00+07:00,5.08,1.00
8,8482511046,00001,BKD1,7123625,ShipCompleted,Đã xuất kho,2026-05-22 14:00:00+07:00,2026-05-21 22:05:29+07:00,2026-05-21 17:00:00+07:00,5.08,1.00
9,8482511025,00001,BKD1,7123607,ShipCompleted,Đã xuất kho,2026-05-22 14:00:00+07:00,2026-05-21 22:05:29+07:00,2026-05-21 17:00:00+07:00,5.08,1.00


**L9.8b · `etd_after_eta` (top 50)**

""


### L9.9 · L8 timestamp — `gio_ra_dock > thoi_gian_di` (ra dock sau khi đi)

Giờ ra dock (`gio_ra_dock`) phải ≤ giờ xe đi (`thoi_gian_di`). Vi phạm thường gặp khi WMS ghi giờ ra dock dựa trên barcode scan SAU khi STM đã chốt giờ đi.

In [58]:
# L9.9 — gio_ra_dock > thoi_gian_di
# (alias đổi tên — `thoi_gian_di` là cột gốc có type khác nhau giữa 2 MV
#  nguồn → để alias trùng tên gây block-structure mismatch khi UNION ALL.)
q("""
SELECT so, orderlinenumber, whseid, customer_code, status, e2e_label,
       toDateTime(delivery_date_1, 'Asia/Ho_Chi_Minh')      AS gi_date,
       toDateTime(gio_ra_dock,     'Asia/Ho_Chi_Minh')      AS ra_dock_time,
       toDateTime(thoi_gian_di,    'Asia/Ho_Chi_Minh')      AS depart_time,
       round(dateDiff('minute', thoi_gian_di, gio_ra_dock) / 60.0, 2)
                                                            AS hours_swapped,
       original_cse, shipped_cse
FROM {T_MAIN}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
  AND gio_ra_dock IS NOT NULL AND thoi_gian_di IS NOT NULL
  AND gio_ra_dock > thoi_gian_di
ORDER BY hours_swapped DESC
LIMIT 100
""")


,so,orderlinenumber,whseid,customer_code,status,e2e_label,gi_date,ra_dock_time,depart_time,hours_swapped,original_cse,shipped_cse
0,8482500682,00001,BKD1,7151496,ShipCompleted,Đã vận chuyển,2026-05-09 14:00:00+07:00,2026-05-11 20:26:14+07:00,2026-05-09 20:00:35+07:00,48.43,96.00,96.00
1,8482495346,00001,BKD1,6840249,ShipCompleted,Đã vận chuyển,2026-05-09 14:00:00+07:00,2026-05-11 07:27:50+07:00,2026-05-09 20:00:45+07:00,35.45,3.00,3.00
2,8482500581,00001,BKD1,6840249,ShipCompleted,Đã vận chuyển,2026-05-09 14:00:00+07:00,2026-05-11 07:27:50+07:00,2026-05-09 20:00:45+07:00,35.45,3.00,3.00
3,8482500581,00002,BKD1,6840249,ShipCompleted,Đã vận chuyển,2026-05-09 14:00:00+07:00,2026-05-11 07:27:50+07:00,2026-05-09 20:00:45+07:00,35.45,10.00,10.00
4,8482500581,00006,BKD1,6840249,ShipCompleted,Đã vận chuyển,2026-05-09 14:00:00+07:00,2026-05-11 07:27:50+07:00,2026-05-09 20:00:45+07:00,35.45,20.00,20.00
5,8482500581,00007,BKD1,6840249,ShipCompleted,Đã vận chuyển,2026-05-09 14:00:00+07:00,2026-05-11 07:27:50+07:00,2026-05-09 20:00:45+07:00,35.45,5.00,5.00
6,8482501362,00001,BKD1,7151496,ShipCompleted,Đã vận chuyển,2026-05-09 14:00:00+07:00,2026-05-11 07:27:50+07:00,2026-05-09 20:00:45+07:00,35.45,24.00,24.00
7,8482501362,00004,BKD1,7151496,ShipCompleted,Đã vận chuyển,2026-05-09 14:00:00+07:00,2026-05-11 07:27:50+07:00,2026-05-09 20:00:45+07:00,35.45,16.00,16.00
8,8482501362,00007,BKD1,7151496,ShipCompleted,Đã vận chuyển,2026-05-09 14:00:00+07:00,2026-05-11 07:27:50+07:00,2026-05-09 20:00:45+07:00,35.45,48.00,48.00
9,8482501362,00009,BKD1,7151496,ShipCompleted,Đã vận chuyển,2026-05-09 14:00:00+07:00,2026-05-11 07:27:50+07:00,2026-05-09 20:00:45+07:00,35.45,23.00,23.00


## L10 · Ad-hoc SQL — slot truy vấn tự do

Sửa `sql` bên dưới rồi chạy cell. `q()` tự thay placeholder `{from_date}` / `{to_date}` / `{date_col}` / `{uom_col}` / `{T_MAIN}` / `{T_FLASH}` / `{T_DROP}`.

Ví dụ: filter 1 SO, 1 whseid, hoặc 1 khoảng giờ:

```sql
SELECT * FROM {T_MAIN}
WHERE so = 'SO-xxxxxx'
ORDER BY orderlinenumber
```

In [59]:
sql = """
SELECT so, orderlinenumber, whseid, customer_code, status, e2e_label,
       toDateTime(delivery_date_1, 'Asia/Ho_Chi_Minh') AS gi_date,
       original_cse, shipped_cse, san_luong_giao_cse
FROM {T_MAIN}
WHERE toDate({date_col}) BETWEEN toDate('{from_date}') AND toDate('{to_date}')
  -- AND so = 'SO-XXXXXX'
  -- AND whseid = 'NKD'
  -- AND e2e_label = 'Đã vận chuyển'
ORDER BY delivery_date_1 DESC, so
LIMIT 100
"""
q(sql)


,so,orderlinenumber,whseid,customer_code,status,e2e_label,gi_date,original_cse,shipped_cse,san_luong_giao_cse
0,8482517234,00002,NKD,7126256,ShipCompleted,Đã vận chuyển,2026-05-28 14:00:00+07:00,5.00,5.00,5.00
1,8482517234,00004,NKD,7126256,ShipCompleted,Đã vận chuyển,2026-05-28 14:00:00+07:00,5.00,5.00,5.00
2,8482517234,00003,NKD,7126256,ShipCompleted,Đã vận chuyển,2026-05-28 14:00:00+07:00,2.00,2.00,2.00
3,8482517234,00005,NKD,7126256,ShipCompleted,Đã vận chuyển,2026-05-28 14:00:00+07:00,3.00,3.00,3.00
4,8482517234,00006,NKD,7126256,ShipCompleted,Đã vận chuyển,2026-05-28 14:00:00+07:00,3.00,3.00,3.00
5,8482517234,00001,NKD,7126256,ShipCompleted,Đã vận chuyển,2026-05-28 14:00:00+07:00,5.00,5.00,5.00
6,8482517235,00003,NKD,7120478,ShipCompleted,Đã vận chuyển,2026-05-28 14:00:00+07:00,3.00,3.00,3.00
7,8482517235,00002,NKD,7120478,ShipCompleted,Đã vận chuyển,2026-05-28 14:00:00+07:00,3.00,3.00,3.00
8,8482517235,00005,NKD,7120478,ShipCompleted,Đã vận chuyển,2026-05-28 14:00:00+07:00,5.00,5.00,5.00
9,8482517235,00001,NKD,7120478,ShipCompleted,Đã vận chuyển,2026-05-28 14:00:00+07:00,3.00,3.00,3.00
